In [3]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [4]:
import sqlite3
import os

db_path = os.path.join('..', 'db.sqlite3')

conn = sqlite3.connect(db_path)

query = """
SELECT *
FROM restaurant_orderitem
JOIN restaurant_order ON restaurant_orderitem.order_id = restaurant_order.id
JOIN restaurant_product ON restaurant_orderitem.product_id = restaurant_product.id
"""

df_initial = pd.read_sql_query(query, conn)
conn.close()


csv_filename = 'combined_restaurant_data.csv'
df_initial.to_csv(csv_filename, index=False)

df_final = pd.read_csv(csv_filename)

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
import matplotlib.pyplot as plt

top_10_products = df_final.groupby('name')['quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 8))
top_10_products.plot(
    kind='pie',
    autopct='%1.1f%%',
    cmap='Set3'
)

plt.title('Top-10')
plt.ylabel('')
plt.tight_layout()

plt.savefig('top_10_popularity.png')
plt.show()

print(top_10_products)

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df_final['item_price_total'] = df_final['price'] * df_final['quantity']

top_10_revenue = df_final.groupby('name')['item_price_total'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 8))
top_10_revenue.plot(
    kind='pie',
    autopct='%1.1f%%',
)

plt.title('Топ-10 продуктів, що приносять найбільший виторг ($)')
plt.ylabel('')
plt.tight_layout()

plt.savefig('top_10_revenue.png')
plt.show()

print(top_10_revenue)

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df_final['order_datetime'] = pd.to_datetime(df_final['datetime'])

df_final['order_hour'] = df_final['order_datetime'].dt.hour

hourly_income = df_final.groupby('order_hour')['item_price_total'].sum()

hourly_income = hourly_income.reindex(range(24), fill_value=0)

plt.figure(figsize=(12, 6))
hourly_income.plot(kind='bar', color='skyblue', edgecolor='navy', alpha=0.8)

plt.title('Загальний дохід ресторану по годинах', fontsize=14)
plt.xlabel('Година доби (0-23)', fontsize=12)
plt.ylabel('Сума продажів ($)', fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df_final['datetime'] = pd.to_datetime(df_final['datetime'])

df_final['day_of_week'] = df_final['datetime'].dt.day_name()

days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

daily_income = df_final.groupby('day_of_week')['item_price_total'].sum().reindex(days_order)

plt.figure(figsize=(10, 6))
daily_income.plot(kind='bar', color='orange', edgecolor='darkred', alpha=0.7)

plt.title('Загальний виторг ресторану за днями тижня', fontsize=14)
plt.xlabel('День тижня', fontsize=12)
plt.ylabel('Сума продажів ($)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('daily_income_bar_chart.png')
plt.show()